# Joint_log_prob_windowed & window_loss test

In [1]:
import sys
sys.path.insert(0, '/home/abethke/Tiberius_Github/Tiberius') # Load Tiberius from Github repo

import tensorflow as tf
from hidten import HMMMode
from tiberius.hmm import HMMBlock
from hidten import HMMMode

from bricks2marble.tf.hmm.tools import left_right_3mers
from tiberius.models import joint_log_prob_windowed, window_loss

2026-05-05 11:51:16.214984: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-05 11:51:16.442448: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-05 11:51:18.850117: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# ---- Step 1: Build HMMBlock ----

mode = HMMMode(0) # 0 = forward log
hiddenmm = HMMBlock(mode, parallel=1, training=False, emitter_epsilon =  None)
# emitter_epsilon=None required to use emission_parameters() with D=5 (LSTM output size)
# instead of emission_parameters_eye() with D=S=15

hiddenmm.build(input_shape=(None, None, 5)) # LSTM has output size D=5

2026-05-05 11:51:22.492635: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
print("n_states:", hiddenmm.config.n_states)
print("Transition matrix shape:", hiddenmm.hmm.transitioner.matrix().shape)
print("Emitter[0] matrix shape:", hiddenmm.hmm.emitter[0].matrix().shape)
print("Emitter[1] matrix shape:", hiddenmm.hmm.emitter[1].matrix().shape)
print("Emitter[2] matrix shape:", hiddenmm.hmm.emitter[2].matrix().shape)

n_states: 15
Transition matrix shape: (1, 15, 15)
Emitter[0] matrix shape: (1, 15, 5)
Emitter[1] matrix shape: (1, 15, 65)
Emitter[2] matrix shape: (1, 15, 65)


In [4]:
# ---- Step 2: Generate test data ----

B, T = 2, 100

# LSTM output (B, T, 5)
x = tf.nn.softmax(tf.random.normal((B, T, 5)), axis=-1)  

# Nucleotide sequences (B, T, 5)
nuc_indices = tf.random.uniform((B, T), minval=0, maxval=4, dtype=tf.int32)
nuc = tf.one_hot(nuc_indices, depth=5)

# Calculate 3-mere 
nuc_left, nuc_right = left_right_3mers(nuc, uniform_N=False) #(B,T,65)

# States (B, T, H)
states = tf.random.uniform((B, T, 1), minval=0, maxval=15, dtype=tf.int32)

observations = (x, nuc_left, nuc_right)

In [5]:
# ---- Step 3: Test functions ----
joint = joint_log_prob_windowed(hiddenmm, *observations, states=states, c=2, d=10) 
print(f"{joint=}") # (B, H) = (2, 1)

loss = window_loss(hiddenmm, *observations, states=states, c=2, d=10) 
print(f"{loss=}") # (B, H) = (2, 1)

2026-05-05 11:51:29.739131: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT32 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


joint=<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[-1.3000009e+09],
       [-1.5000009e+09]], dtype=float32)>
loss=<tf.Tensor: shape=(2, 1), dtype=float32, numpy=
array([[1.2999999e+09],
       [1.4999999e+09]], dtype=float32)>
